# **Kaggle Challenge 3: Đề tài dự đoán thể loại nhạc.**
# **PHẦN 3: TIỀN XỬ LÝ DỮ LIỆU**

## 1. Mô tả vấn đề
+ **Mô tả**:
   - Dự báo thể loại nhạc dựa trên 17 tính chất của dataset.
   - Dữ liệu đầu vào gồm 2 file:
      * train_clean.pkl: Dữ liệu từ file train đã được xử lý ở phần 1.
      * test_clean.pkl: Dữ liệu từ file test đã được xử lý ở phần 1.

+ **Mục tiêu**:
   - Xử lý dữ liệu để các mô hình máy học có thể học hiệu quả, thêm một số đặc trưng giúp cải tiến hiệu suất mô hình.

## 2. Chuẩn bị vấn đề

### 2.1. Import các thư viện cần thiết

In [718]:
# Load libraries
from IPython import display
import numpy as np
import pickle

import matplotlib.pyplot as plt
import random

import pandas as pd
import seaborn as sns
import regex as re

from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler, PowerTransformer, MultiLabelBinarizer
from sklearn.pipeline import Pipeline

# Hiển thị tối đa 200 cột
pd.set_option('display.max_columns', 200)

### 2.2. Đọc dataset train và test từ dataset đã xử lý trong phần dọn dẹp dữ liệu
Đọc các file .pkl trong thư mục /clean/data

In [719]:
train_data = pd.DataFrame(pd.read_pickle("../clean/data/train_clean.pkl"))
test_data = pd.DataFrame(pd.read_pickle("../clean/data/test_clean.pkl"))

### 2.3. Kiểm tra dữ liệu dataset train và test

In [720]:
train_data.head()                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

,Artist Name,Track Name,Popularity,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_in min/ms,time_signature,Class
0,Marina Maximilian,Not Afraid,37.0,0.334,0.536,9.0,-6.649,0,0.0381,0.378000,0.003475,0.106,0.235,152.429,204947.0,4,9
1,The Black Keys,Howlin' for You,67.0,0.725,0.747,11.0,-5.545,1,0.0876,0.027200,0.046800,0.104,0.380,132.921,191956.0,4,6
2,Royal & the Serpent,phuck u,44.0,0.584,0.804,7.0,-6.094,1,0.0619,0.000968,0.635000,0.284,0.635,159.953,161037.0,4,10
3,Detroit Blues Band,Missing You,12.0,0.515,0.308,5.0,-14.711,1,0.0312,0.907000,0.021300,0.300,0.501,172.472,298093.0,3,2
4,Coast Contra,My Lady,48.0,0.565,0.777,6.0,-5.096,0,0.2490,0.183000,0.003475,0.211,0.619,88.311,254145.0,4,5


In [721]:
test_data.head()

,Artist Name,Track Name,Popularity,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_in min/ms,time_signature
0,Juan Pablo Vega,Matando (feat. Vic Mirallas),44.0,0.691,0.670,2.0,-7.093,0,0.0941,0.075700,0.035200,0.1970,0.635,89.965,200000.0,4
1,Kappi Kat,Baarish,14.0,0.461,0.777,2.0,-7.469,1,0.0306,0.388000,0.923000,0.2910,0.525,163.043,283909.0,4
2,Plain White T's,Hey There Delilah,80.0,0.656,0.291,2.0,-10.572,1,0.0293,0.872000,0.002445,0.1140,0.298,103.971,232533.0,4
3,WALK THE MOON,Different Colors,52.0,0.480,0.826,6.0,-4.602,1,0.0397,0.000797,0.000001,0.1250,0.687,96.000,222053.0,4
4,Peled,◊ß◊®◊ô◊ñ,23.0,0.734,0.729,1.0,-6.381,0,0.2830,0.147000,0.004930,0.0672,0.805,76.030,118439.0,4


## 3. Tiền xử lý dữ liệu 

### 3.1. Tách biến mục tiêu khỏi tập train

In [722]:
train_y = train_data["Class"]  # tách biến mục tiêu ra khỏi tập train
train_y.to_pickle("./data/train_y.pkl") # Lưu file.
train_data = train_data.drop(columns="Class") 


### 3.2. Xử lý các biến số có phân bố liên tục
Các biến có phân bố liên tục bao gồm: Popularity, energy, danceability, loudness, speechiness, acousticness, instrumentalness, liveness, valence, tempo, duration_in min/ms. Tuy nhiên một số biến có phân phối trong khoảng $[0, 1]$ nên có thể không cần xử lý thêm.

#### **1) Biến Popularity**
Biến này có phân bố tương đối ổn định nhưng có phân bố lớn $[0, 100]$, có thể chuẩn hóa bằng StandardScaler.

In [723]:
# Chuẩn hóa bằng StandardScaler
scaler = StandardScaler()

train_data[["Popularity"]] = scaler.fit_transform(train_data[["Popularity"]])
test_data[["Popularity"]] = scaler.transform(test_data[["Popularity"]])

#### **2) Biến Loudness**
Biến này có phân bố lệch trái với nhiều giá trị ngoại lai và có phân bố âm, có thể dùng Yeo-Johnson transformer và RobustScaler để xử  lý.

In [724]:
# Tạo pipeline xử lý bằng Yeo-Johnson và Robust Scaler
pipeline = Pipeline([
    ("pt", PowerTransformer(method="yeo-johnson")),
    ("rs", RobustScaler())
])
train_data[["loudness"]] = pipeline.fit_transform(train_data[["loudness"]])
test_data[["loudness"]] = pipeline.transform(test_data[["loudness"]])

#### **3. Biến duration_in min/ms**
Biến này có phân bố lệch phải với nhiều giá trị ngoại lai, có thể dùng log transform và Robust Scaler để xử lý.

In [725]:
train_data["duration_in min/ms"] = train_data["duration_in min/ms"].apply(np.log1p)
test_data["duration_in min/ms"] = test_data["duration_in min/ms"].apply(np.log1p)

scaler = RobustScaler()

train_data[["duration_in min/ms"]] = scaler.fit_transform(train_data[["duration_in min/ms"]])
test_data[["duration_in min/ms"]] = scaler.transform(test_data[["duration_in min/ms"]])

### 3.3. Xử lý các biến số có phân bố rời rạc
Các biến có phân bố rời rạc bao gồm key, mode, time_signature

In [726]:
categorical_data = ["key", "mode", "time_signature"]

numerical_data = train_data.drop(columns=categorical_data).columns

# One-hot encode các cột categorical
encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
encoder.fit(train_data[categorical_data])

train_cat = pd.DataFrame(
    encoder.transform(train_data[categorical_data]),
    columns=encoder.get_feature_names_out(categorical_data),
    index=train_data.index
)
test_cat = pd.DataFrame(
    encoder.transform(test_data[categorical_data]),
    columns=encoder.get_feature_names_out(categorical_data),
    index=test_data.index
)

# Nối lại với phần numeric
train_data = pd.concat([train_data[numerical_data], train_cat], axis=1)
test_data = pd.concat([test_data[numerical_data], test_cat], axis=1)

In [727]:
train_data.head() # Kiểm tra

,Artist Name,Track Name,Popularity,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_in min/ms,key_2.0,key_3.0,key_4.0,key_5.0,key_6.0,key_6.5,key_7.0,key_8.0,key_9.0,key_10.0,key_11.0,mode_1,time_signature_3,time_signature_4,time_signature_5
0,Marina Maximilian,Not Afraid,-0.436403,0.334,0.536,0.086697,0.0381,0.378000,0.003475,0.106,0.235,152.429,-0.045761,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,The Black Keys,Howlin' for You,1.306175,0.725,0.747,0.376282,0.0876,0.027200,0.046800,0.104,0.380,132.921,-0.201026,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0
2,Royal & the Serpent,phuck u,-0.029802,0.584,0.804,0.226773,0.0619,0.000968,0.635000,0.284,0.635,159.953,-0.617447,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
3,Detroit Blues Band,Missing You,-1.888552,0.515,0.308,-1.264658,0.0312,0.907000,0.021300,0.300,0.501,172.472,0.842539,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
4,Coast Contra,My Lady,0.202542,0.565,0.777,0.507956,0.2490,0.183000,0.003475,0.211,0.619,88.311,0.464365,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [728]:
test_data.head()

,Artist Name,Track Name,Popularity,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_in min/ms,key_2.0,key_3.0,key_4.0,key_5.0,key_6.0,key_6.5,key_7.0,key_8.0,key_9.0,key_10.0,key_11.0,mode_1,time_signature_3,time_signature_4,time_signature_5
0,Juan Pablo Vega,Matando (feat. Vic Mirallas),-0.029802,0.691,0.670,-0.018397,0.0941,0.075700,0.035200,0.1970,0.635,89.965,-0.103694,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,Kappi Kat,Baarish,-1.772380,0.461,0.777,-0.103084,0.0306,0.388000,0.923000,0.2910,0.525,163.043,0.726949,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
2,Plain White T's,Hey There Delilah,2.061293,0.656,0.291,-0.687739,0.0293,0.872000,0.002445,0.1140,0.298,103.971,0.253649,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
3,WALK THE MOON,Different Colors,0.434886,0.480,0.826,0.664241,0.0397,0.000797,0.000001,0.1250,0.687,96.000,0.144308,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
4,Peled,◊ß◊®◊ô◊ñ,-1.249606,0.734,0.729,0.153053,0.2830,0.147000,0.004930,0.0672,0.805,76.030,-1.345897,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


### 3.4. Xử lý biến Artist Name và Track Name
Mã hóa các biến này theo nhóm đã làm ở phần 2.

In [729]:
artistCategories = [
    "cherry crimson red ruby scarlet".split(),
    "adult bad big great heavy".split(),
    "death dead destruction dying violent".split(),
    "offspring young youth".split(),
    "baron king power queen rex".split(),
    "black dark inglorious".split(),
    "brother brothers buddy".split(),
    "clean snowy white".split(),
    "blue blues".split()
]

trackCategories = [
	"edition version".split(" "),
	"remaster remastered".split(" "),
	"acoustic".split(" "),
	"remix".split(" "),
	"feat".split(" ")
]

In [730]:
word_to_artist_category = {}
word_to_track_category = {}

for i, group in enumerate(trackCategories):
    for w in group:
        word_to_track_category[w] = i
        

for i, group in enumerate(artistCategories):
    for w in group:
        word_to_artist_category[w] = i
        
# Dọn dẹp chuỗi
def clean_text(text):
    if pd.isna(text):
        return ""
    # Chuyển chữ thường
    text = text.lower()
    # Chỉ giữ các chữ cái Latin và khoảng trắng
    text = re.sub(r'[^\p{L}\s]', '', text)
    # Loại khoảng trắng thừa
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def get_categories_artist_for_row(text):
    words = clean_text(text).lower().split()
    groups = set()
    for w in words:
        if w in word_to_artist_category:
            groups.add(word_to_artist_category[w] + 1)
    return sorted(groups)


def get_categories_track_for_row(text):
    words = clean_text(text).lower().split()
    groups = set()
    for w in words:
        if w in word_to_track_category:
            groups.add(word_to_track_category[w] + 1)
    return sorted(groups)

# Tạo cột phân loại
train_data["CategoryArtist"] = train_data["Artist Name"].apply(get_categories_artist_for_row)
train_data["CategoryTrack"] = train_data["Track Name"].apply(get_categories_track_for_row)
test_data["CategoryArtist"] = test_data["Artist Name"].apply(get_categories_artist_for_row)
test_data["CategoryTrack"] = test_data["Track Name"].apply(get_categories_track_for_row)

In [731]:
train_data["CategoryArtist"].head()

0     []
1    [6]
2     []
3    [9]
4     []
Name: CategoryArtist, dtype: object

In [732]:
train_data["CategoryTrack"].head()

0    []
1    []
2    []
3    []
4    []
Name: CategoryTrack, dtype: object

**Nhận xét:** Dữ liệu mới sau khi nhóm lại có dạng mảng, với mảng rỗng là không thuộc nhóm nào, mảng $[1, 2]$ là thuộc nhóm 1 và 2 (do một câu chứa cả 2 từ khóa có trong mỗi nhóm).

Dùng MultiLabelBinarizer để xử lý dữ liệu có dạng mảng, ví dụ biến có dạng $[0, 2, 3, 0, 1]$ sẽ được mã hóa thành 5 cột mới có giá trị $col_1 = 0, col_2 = 1, col_3 = 1, col_4 = 0, col_5 = 1$

In [733]:
# Tập train
def encode_multilabel(df, col_name):
    mlb = MultiLabelBinarizer()
    encoded = mlb.fit_transform(df[col_name])

    # Tạo các cột mới map theo MultiLabelBinarizer.
    encoded_df = pd.DataFrame(
        encoded,
        columns=[f"{col_name}_{cls}" for cls in mlb.classes_],
        index=df.index
    )

    # Gộp vào dataFrame gốc
    df = pd.concat([df, encoded_df], axis=1)
    df = df.drop(columns=[col_name])
    return df, mlb

# Tập test
def transform_multilabel(df, col_name, mlb):
    encoded = mlb.transform(df[col_name])
    
    encoded_df = pd.DataFrame(
        encoded,
        columns=[f"{col_name}_{cls}" for cls in mlb.classes_],
        index=df.index
    )
    
    df = df.drop(columns=[col_name])
    df = pd.concat([df, encoded_df], axis=1)
    
    return df

train_data, mlb_track = encode_multilabel(train_data, "CategoryTrack")
train_data, mlb_artist = encode_multilabel(train_data, "CategoryArtist")
test_data = transform_multilabel(test_data, "CategoryTrack", mlb_track)
test_data = transform_multilabel(test_data, "CategoryArtist", mlb_artist)

# Xóa các cột Artist Name, Track Name do các cột này đã được mã hóa sang dữ liệu mới
train_data = train_data.drop(columns=["Artist Name", "Track Name"])
test_data = test_data.drop(columns=["Artist Name", "Track Name"])


In [734]:
train_data.head()

,Popularity,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_in min/ms,key_2.0,key_3.0,key_4.0,key_5.0,key_6.0,key_6.5,key_7.0,key_8.0,key_9.0,key_10.0,key_11.0,mode_1,time_signature_3,time_signature_4,time_signature_5,CategoryTrack_1,CategoryTrack_2,CategoryTrack_3,CategoryTrack_4,CategoryTrack_5,CategoryArtist_1,CategoryArtist_2,CategoryArtist_3,CategoryArtist_4,CategoryArtist_5,CategoryArtist_6,CategoryArtist_7,CategoryArtist_8,CategoryArtist_9
0,-0.436403,0.334,0.536,0.086697,0.0381,0.378000,0.003475,0.106,0.235,152.429,-0.045761,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1.306175,0.725,0.747,0.376282,0.0876,0.027200,0.046800,0.104,0.380,132.921,-0.201026,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
2,-0.029802,0.584,0.804,0.226773,0.0619,0.000968,0.635000,0.284,0.635,159.953,-0.617447,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,-1.888552,0.515,0.308,-1.264658,0.0312,0.907000,0.021300,0.300,0.501,172.472,0.842539,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
4,0.202542,0.565,0.777,0.507956,0.2490,0.183000,0.003475,0.211,0.619,88.311,0.464365,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [735]:
test_data.head()

,Popularity,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_in min/ms,key_2.0,key_3.0,key_4.0,key_5.0,key_6.0,key_6.5,key_7.0,key_8.0,key_9.0,key_10.0,key_11.0,mode_1,time_signature_3,time_signature_4,time_signature_5,CategoryTrack_1,CategoryTrack_2,CategoryTrack_3,CategoryTrack_4,CategoryTrack_5,CategoryArtist_1,CategoryArtist_2,CategoryArtist_3,CategoryArtist_4,CategoryArtist_5,CategoryArtist_6,CategoryArtist_7,CategoryArtist_8,CategoryArtist_9
0,-0.029802,0.691,0.670,-0.018397,0.0941,0.075700,0.035200,0.1970,0.635,89.965,-0.103694,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
1,-1.772380,0.461,0.777,-0.103084,0.0306,0.388000,0.923000,0.2910,0.525,163.043,0.726949,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,2.061293,0.656,0.291,-0.687739,0.0293,0.872000,0.002445,0.1140,0.298,103.971,0.253649,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
3,0.434886,0.480,0.826,0.664241,0.0397,0.000797,0.000001,0.1250,0.687,96.000,0.144308,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,-1.249606,0.734,0.729,0.153053,0.2830,0.147000,0.004930,0.0672,0.805,76.030,-1.345897,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 4. Kỹ thuật đặc trưng 

## 5. Xuất dataset train, test đã xử lý

In [736]:
train_data.to_pickle("./data/train_processed.pkl")  # lưu pkl
test_data.to_pickle("./data/test_processed.pkl")  # lưu pkl

## 6. Xuất các file backup vào thư mục exps
Lưu thành file ipynb trong thư mục backup.

In [737]:
from datetime import datetime
import os

# Lấy ngày và giờ
timestamp = datetime.now().strftime("%d-%m_%H-%M")

# Thêm ngày giờ vào notebook
notebook_name = f"features_{timestamp}"

# Chọn thư mục lưu vào
save_dir = "..\\exps\\features"

# Xuất ra file ipynb.
os.system(f"copy preprocessing.ipynb {save_dir}\\{notebook_name}.ipynb")

0

# **Kết thúc**